In [ ]:
%pip install datasets
%pip install dotenv

In [ ]:
# Initial setup
from datasets import load_dataset
import dotenv
import os
import requests
import pprint
from datetime import datetime
import csv, json

dotenv.load_dotenv()
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
ALINIA_STAGING_API_KEY = os.getenv("ALINIA_STAGING_API_KEY")

# Create errors folder if needed
if not os.path.exists("./errors"):
    os.makedirs('./errors')

# Helper Classes

In [ ]:
class APIException(Exception):
    def __init__(self, input_json: dict, status_code: int, response_text: str):
        self.input_json = input_json
        self.status_code = status_code
        self.response_text = response_text
    
    def __str__(self) -> str:
        return f"API Error:\nstatus code: {self.status_code}\nresponse text: {self.response_text}\ninput json:{self.input_json}"

class Results:
    def __init__(self):
        self.tp = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        self.accuracy = 0
        self.precision = 0
        self.recall = 0
        self. f1_score = 0
    
    def calculate_stats(self) -> None:
        if self.tp > 0 and self.fp > 0:
            self.accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn)
        else:
            self.accuracy = None

        if self.tp > 0:
            self.precision = self.tp / (self.tp + self.fp)
            self.recall = self.tp / (self.tp + self.fn)
        else:
            self.precision = None
            self.recall = None

        if self.precision is not None and self.recall is not None:
            self.f1_score = 2 * (self.precision * self.recall) / (self.precision + self.recall)
        else:
            self.f1_score = None

    def __str__(self) -> str:
        return f"tp: {self.tp}" + "\n" + f"tn: {self.tn}" + "\n" + f"fp: {self.fp}"+ "\n" + f"fn: {self.fn}"+ "\n" + f"accuracy: {self.accuracy}"+ "\n" + f"precision: {self.precision}"+ "\n" + f"recall: {self.recall}"+ "\n" + f"f1_score: {self.f1_score}"

# Helper Functions

In [ ]:
# Helper functions
def get_detection_config_json(
        violence: bool = False, 
        hate: bool = False, 
        wrongdoing: bool = False,
        sexual: bool = False,
        adversarial: bool = False) -> dict:
    
    detection_config_json = {}

    # Set the safety guard flags
    if violence or hate or wrongdoing or sexual:
        detection_config_json["safety"] = {}

        if violence: detection_config_json["safety"]["violence"] = True
        if hate:  detection_config_json["safety"]["hate"] = True
        if wrongdoing:  detection_config_json["safety"]["wrongdoing"] = True
        if sexual:  detection_config_json["safety"]["sexual"] = True
    
    if adversarial:
        detection_config_json["security"] = {}

        if adversarial:
            detection_config_json["security"]["adversarial"] = True
            
    return detection_config_json

def get_alinia_input_json(input_str: str, detection_config: dict) -> dict:
    return {
        "input": input_str,
        "detection_config": detection_config
    }

def evaluate(input_json : dict) -> dict:
    response = requests.post("https://staging.api.alinia.ai/moderations/",
        headers = {"Authorization": f"Bearer {ALINIA_STAGING_API_KEY}"},
        json = input_json,
    )
    if not response.ok:
        print(f">>>> HTTP error {response.status_code}: {response.text}")
        raise APIException(input_json, response.status_code, response.text)
    try:
        return response.json()
    except requests.exceptions.JSONDecodeError:
        print(f">>>> Failed to decode JSON response (status {response.status_code}): {response.text!r}")
        raise APIException(input_json, response.status_code, response.text)
    
def evaluate_example(example, text_parameter, class_parameter, positive_class, negative_class, detection_config, results_obj, error_path):
    input_str = example[text_parameter]
    label = example[class_parameter]
    response_json = None

    print(f"Evaluating: {input_str}")
    input_json = get_alinia_input_json(input_str, detection_config)
    try:
        response_json = evaluate(input_json)
    except APIException as e:
        print(e)
        with open(error_path, "a") as errors_fp:
            errors_fp.write(str(datetime.now()) + "\n" + str(e) + "\n")
    
    is_flagged = False

    if response_json is not None and 'result' in response_json.keys():
        if 'flagged' in response_json['result'].keys():
            is_flagged = response_json['result']['flagged']

    if response_json is not None:
        if is_flagged and label == positive_class:
            results_obj.tp += 1
        elif is_flagged and label == negative_class:
            results_obj.fp += 1
        elif not is_flagged and label == negative_class:
            results_obj.tn += 1
        elif not is_flagged and label == positive_class:
            results_obj.fn += 1

# This function is currently unused
def evaluate_example_multiclass(example, text_parameter, class_parameter, positive_class_arr, negative_class_arr, detection_config, results_obj, error_path):
    input_str = example[text_parameter]
    label = example[class_parameter]
    response_json = None

    print(f"Evaluating: {input_str}")
    input_json = get_alinia_input_json(input_str, detection_config)
    try:
        response_json = evaluate(input_json)
    except APIException as e:
        print(e)
        with open(error_path, "a") as errors_fp:
            errors_fp.write(str(datetime.now()) + "\n" + str(e) + "\n")
    
    is_flagged = False

    if response_json is not None and 'result' in response_json.keys():
        if 'flagged' in response_json['result'].keys():
            is_flagged = response_json['result']['flagged']

    if response_json is not None:
        if is_flagged and label in positive_class_arr:
            results_obj.tp += 1
        elif is_flagged and label in negative_class_arr:
            results_obj.fp += 1
        elif not is_flagged and label in negative_class_arr:
            results_obj.tn += 1
        elif not is_flagged and label in positive_class_arr:
            results_obj.fn += 1

def calculate_stats_and_save(results_obj, title, filepath):
    # Calculate stats and output results
    results_obj.calculate_stats()
    print(f"{title}:")
    print(results_obj)

    # Save output to file
    with open(filepath, "w") as results_fp:
        results_fp.write(f"{title}:\n")
        results_fp.write(str(results_obj))

In [ ]:
# Create configs
HATE_CONFIG = get_detection_config_json(hate = True)
SEXUAL_CONFIG = get_detection_config_json(sexual = True)
VIOLENCE_CONFIG = get_detection_config_json(violence = True)
WRONGDOING_CONFIG = get_detection_config_json(wrongdoing = True)
ALL_SAFETY_CONFIG = get_detection_config_json(hate = True, sexual = True, violence = True, wrongdoing = True)

ADVERSARIAL_CONFIG = get_detection_config_json(adversarial= True)

# Safety Guard: Hate

In [ ]:
# Hate Speech Public Evaluations

# Doesn't use general version since we're populating 2 results per call.
def evaluate_hate_1(example, detection_config, results, results2):
    input_str = example["tweet"]
    label = example["class"]

    print(f"Evaluating: {input_str}")
    input_json = get_alinia_input_json(input_str, detection_config)
    response_json = evaluate(input_json)
    
    errors = []
    is_flagged = False

    if 'result' in response_json.keys():
        if 'flagged' in response_json['result'].keys():
            is_flagged = response_json['result']['flagged']
        else:
            errors.append(f'>>>> Missing result, flagged for: {example}')
            print(f'>>>> Missing result.flagged for: {example}')
    else:
        errors.append(f'>>>> Missing result for: {example}')
        print(f'>>>> Missing result for: {example}')

    # pprint.pp(response_json)
    # print(f"is_flagged: {is_flagged}, label: {label}")

    # Only Hate class as positive label
    if is_flagged and label == 0:
        results.tp += 1
    elif is_flagged and label > 0:
        results.fp += 1
    elif not is_flagged and label > 0:
        results.tn += 1
    elif not is_flagged and label == 0:
        results.fn += 1

    # Either Hate or Offensive class as positive label
    if is_flagged and (label == 0 or label == 1):
        results2.tp += 1
    elif is_flagged and label == 2:
        results2.fp += 1
    elif not is_flagged and label == 2:
        results2.tn += 1
    elif not is_flagged and (label == 0 or label == 1):
        results2.fn += 1

In [ ]:
# ===== Hate Evaluation 1 =====

# Dataset details
# id: id
# count: number of annotators
# hate_speech_count: number of annotators who judged the tweet to be hate speech
# offensive_language_count: number of annotators who judged the tweet to be offensive
# neither_count: number of users who judged the tweet to be neither offensive nor non-offensive
# class: class label for majority of annotators (0: 'hate-speech', 1: 'offensive-language' or 2: 'neither')
# tweet: The content of the tweet
dataset_hate_1 = load_dataset("tdavidson/hate_speech_offensive", token = HUGGINGFACE_API_KEY)

results_hate_1a = Results()
results_hate_1b = Results()

print("Evaluating tdavidson/hate_speech_offensive...")

# Small sample test
# small_ds.map(lambda example: evaluate_hate_1(example, HATE_CONFIG, results_hate_1, results_hate_2))
# dataset_hate_1.map(lambda example: evaluate_hate_1(example, HATE_CONFIG, results_hate_1, results_hate_2))

# Full test
dataset_hate_1.map(lambda example: evaluate_hate_1(example, HATE_CONFIG, results_hate_1a, results_hate_1b))

# Special case of saving two class interpretations (with and without offensive language)
# so I don't use the helper functions here
results_hate_1a.calculate_stats()
print("Results (Hate Label Only):")
print(results_hate_1a)

results_hate_1b.calculate_stats()
print("Results (Hate + Offensive Language Labels):")
print(results_hate_1b)

# Save output to file
with open("./results/results-hate-1.txt", "w") as results_fp:
    results_fp.write("Results (Hate Label Only):\n")
    results_fp.write(str(results_hate_1a))
    results_fp.write("\n\nResults (Hate + Offensive Language Labels):\n")
    results_fp.write(str(results_hate_1b))

In [ ]:
# ===== Hate Evaluation 2 =====
# Dataset details
# (unnamed): row id
# text: the content text
# label: class label (0: non-hate, 1: hate)

dataset_hate_2_en = load_dataset("FrancophonIA/multilingual-hatespeech-dataset", "English_test", token = HUGGINGFACE_API_KEY)
dataset_hate_2_fr = load_dataset("FrancophonIA/multilingual-hatespeech-dataset", "French_test", token = HUGGINGFACE_API_KEY)
dataset_hate_2_es = load_dataset("FrancophonIA/multilingual-hatespeech-dataset", "Spain_test", token = HUGGINGFACE_API_KEY)

results_hate_2_en = Results()
results_hate_2_fr = Results()
results_hate_2_es = Results()

# ----- Test English examples -----
print("Evaluating FrancophonIA/multilingual-hatespeech-dataset (English)...")

# Sample sample test
# small_ds = dataset_hate_2_en["test"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_en, "./errors/errors-hate-2.txt"))

# Full test
dataset_hate_2_en.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_en, "./errors/errors-hate-2.txt"))
calculate_stats_and_save(results_hate_2_en, "Results Multilingual Hate Speech (English)", "./results/results-hate-2-en.txt")

# ----- Test French examples -----
print("Evaluating FrancophonIA/multilingual-hatespeech-dataset (French)...")

# Sample sample test
# small_ds = dataset_hate_2_fr["test"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_fr, "./errors/errors-hate-2.txt"))

#Full test
dataset_hate_2_fr.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_fr, "./errors/errors-hate-2.txt"))
calculate_stats_and_save(results_hate_2_fr, "Results Multilingual Hate Speech (French)", "./results/results-hate-2-fr.txt")

# ----- Test Spanish examples -----
print("Evaluating FrancophonIA/multilingual-hatespeech-dataset (Spanish)...")

# Small sample test
# small_ds = dataset_hate_2_es["test"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_es, "./errors/errors-hate-2.txt"))

# Full test
dataset_hate_2_es.map(lambda example: evaluate_example(example, "text", "label", 1, 0, HATE_CONFIG, results_hate_2_es, "./errors/errors-hate-2.txt"))
calculate_stats_and_save(results_hate_2_es, "Results Multilingual Hate Speech (Spanish)", "./results/results-hate-2-es.txt")


# Safety Guard: Violence

In [ ]:
# ===== Violence Evaluation 1 =====
# Dataset details:
# id: tweet identifier
# text: the tweet's text
# label: the class label ("VIOLENTO" = violent, "NOVIOLENTO" = non-violent)
# ...

results_violence_1_es = Results()

print("Evaluating VILLANOS (Spanish)...")

# Full test
with open("./datasets/villanos.tsv", "r") as dataset_violence_1_fp:
    reader = csv.DictReader(dataset_violence_1_fp, delimiter="\t")
    for example in reader:
        evaluate_example(example, "text", "label", "VIOLENTO", "NOVIOLENTO", VIOLENCE_CONFIG, results_violence_1_es, "./errors/errors-violence-es.txt")

calculate_stats_and_save(results_violence_1_es, "Results Violent Speech - VILLANOS (Spanish)", "./results/results-violence-1-es.txt")


# Safety Guard: General

In [ ]:
# ===== General Harm Evaluation 1 =====
# Dataset details:
# prompt: the text content of the prompt
# context: additional context
# category: 6 classes identifying the kind of harm (All classes are harmful)
#  (misinformation_disinformation, chemical_biological, cybercrime_intrusion, harassment_bullying, harmful, illegal)

dataset_safety_1_en = load_dataset("walledai/HarmBench", "standard", token = HUGGINGFACE_API_KEY)
results_safety_1_en = Results()

print("Evaluating alledai/HarmBench...")

# Small sample test
# small_ds = dataset_safety_1_en["train"].select(range(100))
# small_ds.map(lambda example: evaluate_example_multiclass(example, "prompt", "category", ["misinformation_disinformation", "chemical_biological", "cybercrime_intrusion", "harassment_bullying", "harmful", "illegal"], 0, ALL_SAFETY_CONFIG, results_safety_1_en, "./errors/errors-safety-1.txt"))

# Full test
dataset_safety_1_en.map(lambda example: evaluate_example_multiclass(example, "prompt", "category", ["misinformation_disinformation", "chemical_biological", "cybercrime_intrusion", "harassment_bullying", "harmful", "illegal"], 0, ALL_SAFETY_CONFIG, results_safety_1_en, "./errors/errors-safety-1.txt"))
calculate_stats_and_save(results_safety_1_en, "Results HarmBench (English)", "./results/results-safety-1-en.txt")

# Security Guard: Adversarial

In [ ]:
# ===== Adversarial Evaluation 1 =====
# Dataset details:
# prompt: The adversarial or benign prompt
# type: The class label ("jailbreak": prompt is a jailbreak, "benign": prompt is benign)

dataset_adversarial_1_en = load_dataset("jackhhao/jailbreak-classification", token = HUGGINGFACE_API_KEY)
results_adversarial_1_en = Results()

print("Evaluating jackhhao/jailbreak-classification (English)...")

# Small sample test
# small_ds = dataset_adversarial_1_en["train"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "prompt", "type", "jailbreak", "benign", ADVERSARIAL_CONFIG, results_adversarial_1_en, "./errors/errors-adversarial-1.txt"))

# Full test
dataset_adversarial_1_en.map(lambda example: evaluate_example(example, "prompt", "type", "jailbreak", "benign", ADVERSARIAL_CONFIG, results_adversarial_1_en, "./errors/errors-adversarial-1.txt"))
calculate_stats_and_save(results_adversarial_1_en, "Results In the Wild Jailbreak (English)", "./results/results-adversarial-1-en.txt")


In [ ]:
# ===== Adversarial Evaluation 2 =====
# Dataset details:
# text: The adversarial or benign prompt
# label: The class label (1: prompt is malicious, 0: prompt is benign)
# category: The type of attack
# source: sample origin
# severity: severity of attack
# group_id: used to track variants
# augmented: true if synthetic
# tags: metadata tags

dataset_adversarial_2_en = load_dataset("neuralchemy/Prompt-injection-dataset", "core", token = HUGGINGFACE_API_KEY)
results_adversarial_2_en = Results()

# Test English examples
print("Evaluating neuralchemy/Prompt-injection-dataset (English)...")

# Small scale test
# small_ds = dataset_adversarial_2_en["train"].select(range(100))
# small_ds.map(lambda example: evaluate_example(example, "text", "label", 1, 0, ADVERSARIAL_CONFIG, results_adversarial_2_en, "./errors/errors-adversarial-2.txt"))

# Full test
dataset_adversarial_2_en.map(lambda example: evaluate_example(example, "text", "label", 1, 0, ADVERSARIAL_CONFIG, results_adversarial_2_en, "./errors/errors-adversarial-2.txt"))
calculate_stats_and_save(dataset_adversarial_2_en, "Results NeuralAlchemy Prompt Injection Dataset", "./results/results-adversarial-2-en.txt")


# Testing out Phare Dataset (IGNORE FOR NOW)

In [ ]:
# ===== Phare Evaluation 1 =====
# It is unclear what the labels are for phare, but I wanted to try something regardless, so I assumed the examples were all positive.

def get_text_from_phare_example(example):
    json_str = example["generations"]
    json_arr = json.loads(json_str)
    message = json_arr[0]["messages"][0]["content"]
    return {"text": message, "label": 1}

dataset_phare_1a = load_dataset("giskardai/phare", "jailbreak_encoding", token = HUGGINGFACE_API_KEY)
dataset_phare_1a_en = dataset_phare_1a.filter(lambda example: example["language"] == "en")
dataset_phare_1a_fr = dataset_phare_1a.filter(lambda example: example["language"] == "fr")
dataset_phare_1a_es = dataset_phare_1a.filter(lambda example: example["language"] == "es")

results_phare_1a_en = Results()
results_phare_1a_fr = Results()
results_phare_1a_es = Results()

# Phare evaluation English
print("Evaluating Phare - Jailbreak Encoding (English)...")
dataset_phare_1a_en.map(lambda example: evaluate_example(get_text_from_phare_example(example), "text", "label", 1, 0, ADVERSARIAL_CONFIG, results_phare_1a_en, "./errors/errors-adversarial-1a.txt"))

results_phare_1a_en.calculate_stats()
print("Results Phare - Jailbreak Encoding (English):")
print(results_phare_1a_en)

# Save output to file
with open("./results/results-phare-1a-en.txt", "w") as results_fp:
    results_fp.write("Results Phare - Jailbreak Encoding (English):\n")
    results_fp.write(str(results_phare_1a_en))

# Phare evaluation French
print("Evaluating Phare - Jailbreak Encoding (French)...")
dataset_phare_1a_fr.map(lambda example: evaluate_example(get_text_from_phare_example(example), "text", "label", 1, 0, ADVERSARIAL_CONFIG, results_phare_1a_fr, "./errors/errors-adversarial-1a.txt"))

results_phare_1a_fr.calculate_stats()
print("Results Phare - Jailbreak Encoding (French):")
print(results_phare_1a_fr)

# Save output to file
with open("./results/results-phare-1a-fr.txt", "w") as results_fp:
    results_fp.write("Results Phare - Jailbreak Encoding (French):\n")
    results_fp.write(str(results_phare_1a_fr))

# Phare evaluation Spanish
print("Evaluating Phare - Jailbreak Encoding (Spanish)...")
dataset_phare_1a_es.map(lambda example: evaluate_example(get_text_from_phare_example(example), "text", "label", 1, 0, ADVERSARIAL_CONFIG, results_phare_1a_es, "./errors/errors-adversarial-1a.txt"))

results_phare_1a_es.calculate_stats()
print("Results Phare - Jailbreak Encoding (Spanish):")
print(results_phare_1a_es)

# Save output to file
with open("./results/results-phare-1a-es.txt", "w") as results_fp:
    results_fp.write("Results Phare - Jailbreak Encoding (Spanish):\n")
    results_fp.write(str(results_phare_1a_es))